# MoMo Fraud Detection — EDA
**Step 1 of 6**

## 0. Install Dependencies

In [ ]:
import subprocess, sys
for pkg in ['pandas','numpy','matplotlib','seaborn']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print('Ready')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format','{:.2f}'.format)

## 2. Load Data

In [ ]:
df = pd.read_csv('Pay-sim.csv')
print(df.shape)
df.head()

## 3. Overview

In [ ]:
print(df.dtypes)
df.describe()

## 4. Missing Values

In [ ]:
df.isnull().sum()

## 5. Duplicates

In [ ]:
print('Duplicates:', df.duplicated().sum())

## 6. Target — isFraud

In [ ]:
vc = df['isFraud'].value_counts()
print(vc)
vc.plot(kind='bar', color=['steelblue','crimson'], title='isFraud Distribution')
plt.show()

## 7. Transaction Type

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4))
df['type'].value_counts().plot(kind='bar',ax=ax[0],color='steelblue',title='All Transactions by Type')
df[df['isFraud']==1]['type'].value_counts().plot(kind='bar',ax=ax[1],color='crimson',title='Fraud by Type')
plt.tight_layout()
plt.show()
print(df.groupby('type')['isFraud'].mean().sort_values(ascending=False).apply(lambda x:f'{x*100:.2f}%'))

## 8. Amount Analysis

In [ ]:
print(df['amount'].describe())
print('Zero amounts:',( df['amount']==0).sum())
fig,ax=plt.subplots(1,2,figsize=(13,4))
ax[0].hist(np.log1p(df['amount']),bins=50,color='steelblue',edgecolor='black')
ax[0].set_title('Amount Distribution (log scale)')
ax[1].hist(np.log1p(df[df['isFraud']==0]['amount']),bins=50,alpha=0.6,color='steelblue',label='Non-Fraud')
ax[1].hist(np.log1p(df[df['isFraud']==1]['amount']),bins=50,alpha=0.6,color='crimson',label='Fraud')
ax[1].legend()
ax[1].set_title('Fraud vs Non-Fraud Amount')
plt.tight_layout()
plt.show()

## 9. Balance Analysis

In [ ]:
print('Zero oldbalanceOrg:',(df['oldbalanceOrg']==0).sum())
print('Zero oldbalanceDest:',(df['oldbalanceDest']==0).sum())
err_orig=(df['newbalanceOrig']+df['amount']-df['oldbalanceOrg']).abs()
err_dest=(df['oldbalanceDest']+df['amount']-df['newbalanceDest']).abs()
print('Balance errors (origin):',(err_orig>1).sum())
print('Balance errors (dest):',(err_dest>1).sum())

## 10. Time Analysis

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(13,7))
df.groupby('step').size().plot(ax=axes[0],color='steelblue',title='All Transactions Over Time')
df[df['isFraud']==1].groupby('step').size().plot(ax=axes[1],color='crimson',title='Fraud Over Time')
plt.tight_layout()
plt.show()

## 11. Outliers

In [ ]:
cols=['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']
fig,axes=plt.subplots(1,5,figsize=(18,4))
for i,c in enumerate(cols):
    axes[i].boxplot(df[c])
    axes[i].set_title(c,fontsize=8)
plt.suptitle('Outlier Box Plots')
plt.tight_layout()
plt.show()

## 12. Correlation

In [ ]:
corr=df[['step','amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','isFraud']].corr()
plt.figure(figsize=(9,6))
sns.heatmap(corr,annot=True,fmt='.2f',cmap='coolwarm',center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 13. Fraud Patterns by Type

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
avg=df.groupby(['type','isFraud'])['amount'].mean().unstack()
avg.columns=['Non-Fraud','Fraud']
avg.plot(kind='bar',ax=axes[0],color=['steelblue','crimson'],title='Avg Amount by Type')
(df.groupby('type')['isFraud'].mean()*100).sort_values(ascending=False).plot(kind='bar',ax=axes[1],color='crimson',title='Fraud Rate % by Type')
plt.tight_layout()
plt.show()

## 14. isFlaggedFraud Check

In [ ]:
print(pd.crosstab(df['isFlaggedFraud'],df['isFraud']))
print('Flagged:',df['isFlaggedFraud'].sum(),'| Actual fraud:',df['isFraud'].sum())